In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, text
import os
import numpy as np

In [ ]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f'SELECT * FROM "{schema}"."{table_name}"')

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    
def split_by_grid(gdf_input, gdf_grid):
    # Optional but VERY important for speed
    # (Shapely 2 / PyGEOS backend)
    gdf = gdf_input.copy()
    grid = gdf_grid.copy()

    gdf_geometry_name = gdf.geometry.name
    grid_geometry_name = grid.geometry.name

    # Make geometries valid (prevents topology errors & slowdowns)
    gdf[gdf_geometry_name] = gdf.make_valid()
    grid[grid_geometry_name] = grid.make_valid()

    # Keep only needed columns
    grid = grid[["tile_index", grid_geometry_name]]   # cell_id = your grid unique ID

    # ---- SPLIT coastline by grid (spatial-index accelerated) ----
    split = gpd.overlay(
        gdf,
        grid,
        how="intersection",
        keep_geom_type=True
    )

    # ---- DISSOLVE per grid cell ----
    result = split.dissolve(
        by="tile_index",
        as_index=False
    )

    return result


def publish_vector_layer(gdf, db_name, user, password, host, port,
                         table_name, schema='public', geom_col='geom'):
    """
    Publishes a GeoDataFrame to a PostGIS-enabled PostgreSQL database.

    Parameters:
    - gdf (gpd.GeoDataFrame): GeoDataFrame to upload.
    - db_name (str): Database name.
    - user (str): Username.
    - password (str): Password.
    - host (str): Host address.
    - port (int): Port number.
    - table_name (str): Target table name.
    - schema (str): Schema (default 'public').
    - geom_col (str): Geometry column name (default 'geom').

    Returns:
    - None
    """

    try:
        # Connection string
        conn_str = f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        # Ensure geometry column name matches
        if gdf.geometry.name != geom_col:
            gdf = gdf.rename_geometry(geom_col)

        # Write to PostGIS
        gdf.to_postgis(
            name=table_name,
            con=engine,
            schema=schema,
            if_exists='fail',
            index=True
        )

        print(f"Successfully published {table_name} ({len(gdf)} features)")

    except Exception as e:
        print(f"Error publishing vector layer: {e}")

In [ ]:
# gdf_1d_grid = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.233',
#     port=5555,
#     table_name='global_klab_1d_tiles',
#     schema='klab_grids'
# )

# gdf_3d_grid = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.233',
#     port=5555,
#     table_name='global_klab_3d_tiles',
#     schema='klab_grids'
# )


# osm_coast_gdf = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.100',
#     port=5555,
#     table_name='global_osm_coastline_4326',
#     schema='public'
# )

un_gdf = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.100',
    port=5555,
    table_name='administrative_units_un_gadm_level0_1d_tile_index_v1',
    schema='public'
)

In [ ]:
joined = gpd.sjoin(un_gdf, un_gdf, predicate="touches")

# Count neighbors per tile
neighbor_counts = joined.groupby(joined.index).size()

# Add count to original grid
un_gdf["n_neighbors"] = neighbor_counts
un_gdf["n_neighbors"] = un_gdf["n_neighbors"].fillna(0)

In [ ]:
# --- Select edge tiles
# Interior tiles usually have max neighbors (e.g. 8 in a square grid)
un_gdf["tile_type"] = np.where(
    un_gdf["n_neighbors"] == 8,
    "continental",
    "coastal"
)

In [ ]:
un_gdf_valid = un_gdf.copy()
un_gdf_valid["geom"] = un_gdf_valid.make_valid()

In [ ]:
un_gdf_valid.to_file("administrative_units_un_gadm_level0_1d_tile_index_v2.shp", driver='ESRI Shapefile')

In [ ]:
publish_vector_layer(un_gdf_valid, 
                     db_name="geoserver", 
                     user= "geoserver", 
                     password= "geoserver", 
                     host='192.168.250.100', 
                     port=5555,
                     table_name="administrative_units_un_gadm_level0_1d_tile_index_v2", 
                     schema='public', 
                     geom_col='geom')